In [ ]:
import os
from pathlib import Path
import hashlib
import json
import re

import pandas as pd
import numpy as np

### Combine all json files together in CSV

Base files from [FelixDrinkall/financial-news-dataset](https://github.com/FelixDrinkall/financial-news-dataset) are placed in \data folder

In [ ]:
data_dir = Path("../data")
years = range(2017, 2024)

dfs = []
for year in years:
    path = data_dir / f"{year}_processed.json"

    with open(path, "r") as f:
        records = json.load(f)

    df = pd.DataFrame(records)
    df["year"] = year  # add source year column for traceability

    dfs.append(df)
    print(f"Loaded {year}: {len(df):,} rows")

combined = pd.concat(dfs, ignore_index=True)
print(f"Total rows: {len(combined):,}")

# Serialize nested columns to JSON strings so CSV stays flat
nested_cols = [
    "named_entities",
    "mentioned_companies",
    "related_companies",
    "industries",
    "authors",
    "sentiment",
    "emotion",
]

for col in nested_cols:
    if col in combined.columns:
        combined[col] = combined[col].apply(
            lambda x: json.dumps(x) if x is not None else None
        )

In [ ]:
out_path = data_dir / "combined_2017_2023.csv"
combined.to_csv(out_path, index=False)
print(f"Saved to {out_path}")

In [ ]:
print(f"Shape: {combined.shape[0]:,} rows × {combined.shape[1]} columns")
print(f"\nYear distribution:")
print(combined["year"].value_counts().sort_index())


missing = combined.isnull().sum()
missing_pct = (missing / len(combined) * 100).round(2)
missing_df = pd.DataFrame({"missing_count": missing, "missing_%": missing_pct})
missing_df = missing_df[missing_df["missing_count"] > 0].sort_values(
    "missing_%", ascending=False
)
print(f"\nColumns with missing values ({len(missing_df)} of {combined.shape[1]}):")
print(missing_df.to_string())


dupes = combined.duplicated(subset=["url"]).sum()
print(f"\nDuplicate URLs: {dupes:,} ({dupes/len(combined)*100:.2f}%)")


print(f"\nData types:\n{combined.dtypes.value_counts()}")


print(f"\ndate_publish range:")
dates = pd.to_datetime(combined["date_publish"], errors="coerce")
print(f"  min: {dates.min()}  max: {dates.max()}")
print(f"  unparseable: {dates.isna().sum():,}")

print(f"\nmaintext length (chars):")
text_len = combined["maintext"].dropna().str.len()
print(text_len.describe().apply(lambda x: f"{x:,.0f}"))

print(f"\nlanguage distribution (top 10):")
print(combined["language"].value_counts().head(10))


def count_parse_errors(series, col_name):
    errors = 0
    for val in series.dropna():
        try:
            json.loads(val)
        except (json.JSONDecodeError, TypeError):
            errors += 1
    print(f"  {col_name}: {errors:,} parse errors out of {series.notna().sum():,}")


print("\nNested JSON field validity:")
for col in ["sentiment", "emotion", "named_entities", "mentioned_companies"]:
    if col in combined.columns:
        count_parse_errors(combined[col], col)


def sentiment_sum_check(series):
    totals = []
    for val in series.dropna().sample(min(1000, len(series)), random_state=42):
        try:
            d = json.loads(val)
            totals.append(sum(d.values()))
        except Exception:
            pass
    totals = np.array(totals)
    off = (np.abs(totals - 1.0) > 0.01).sum()
    print(f"  sentiment scores not summing to 1.0: {off}/{len(totals)} sampled")


if "sentiment" in combined.columns:
    sentiment_sum_check(combined["sentiment"])


print("\n── Summary ──")
print(f"Total articles : {len(combined):,}")
print(f"Unique URLs    : {combined['url'].nunique():,}")
print(f"Columns        : {combined.shape[1]}")
print(
    f"Est. CSV size  : ~{combined.memory_usage(deep=True).sum() / 1e9:.2f} GB in memory"
)

In [ ]:
data_dir = Path("../data")
combined = pd.read_csv(rf"{data_dir}/combined_2017_2023.csv")

### Clean up columns

In [ ]:
# THEMES = {
#     "AI": [
#         r"artificial intelligence",
#         r"machine learning",
#         r"deep learning",
#         r"neural network",
#         r"large language model",
#         r"\bllm\b",
#         r"generative ai",
#         r"chatgpt",
#         r"openai",
#         r"gpt-\d",
#         r"natural language processing",
#         r"reinforcement learning",
#         r"transformer model",
#         r"foundation model",
#         r"diffusion model",
#         r"stable diffusion",
#         r"midjourney",
#         r"github copilot",
#     ],
#     "Covid19": [
#         r"covid",
#         r"covid.19",
#         r"coronavirus",
#         r"sars.cov.2",
#         r"covid.*lockdown|lockdown.*covid",
#         r"covid.*quarantine|quarantine.*covid",
#         r"covid.*vaccin|vaccin.*covid",
#         r"corona.*vaccin",
#         r"pfizer.*vaccine",
#         r"pfizer.*biontech",
#         r"pfizer.*covid",
#         r"moderna.*vaccine",
#         r"moderna.*covid",
#         r"moderna.*mrna",
#         r"astrazeneca.*vaccine",
#         r"astrazeneca.*covid",
#         r"omicron",
#         r"delta variant",
#         r"social distancing",
#         r"ppe shortage",
#         r"covid.*ventilator|ventilator.*covid",
#         r"covid.*public health emergency",
#     ],
#     "USA_Tariff": [
#         r"us tariff",
#         r"u\.s\. tariff",
#         r"american tariff",
#         r"us.*trade war|trade war.*china|trump.*trade war|biden.*trade war",
#         r"section 301",
#         r"section 232",
#         r"trump tariff",
#         r"biden tariff",
#         r"china tariff",
#         r"us.*steel tariff|trump.*steel tariff|american.*steel tariff",
#         r"us.*aluminum tariff|trump.*aluminum tariff|american.*aluminum tariff",
#         r"retaliatory tariff",
#         r"us.*wto|wto.*us",
#     ],
#     "Russia_Ukraine": [
#         r"russia.*ukraine|ukraine.*russia",
#         r"russian invasion",
#         r"kyiv",
#         r"kiev",
#         r"zelensky",
#         r"zelenskyy",
#         r"putin.*ukraine",
#         r"ukraine war",
#         r"ukraine conflict",
#         r"nato.*ukraine",
#         r"ukrainian military",
#         r"donbas",
#         r"crimea.*russia|russia.*crimea|crimea.*ukraine|ukraine.*crimea|annexation.*crimea",
#         r"mariupol",
#         r"russian military.*ukraine|ukraine.*russian military",
#         r"sanctions.*russia|russia.*sanctions",
#         r"grain corridor",
#         r"nord stream",
#         r"kherson",
#         r"zaporizhzhia",
#     ],
#     "Oil_Crisis": [
#         r"oil crisis",
#         r"oil price spike",
#         r"oil shortage",
#         r"fuel crisis",
#         r"oil embargo",
#         r"petroleum supply",
#         r"oil production cut",
#         r"energy supply chain",
#         r"oil price.*surge",
#         r"oil price.*crash",
#         r"oil price.*shock",
#         r"opec.*production cut|production cut.*opec",
#         r"oil supply disruption|supply disruption.*oil",
#         r"oil.*energy crisis|energy crisis.*oil",
#     ],
#     "Semiconductor": [
#         r"chip shortage",
#         r"chip crisis",
#         r"chip war",
#         r"silicon wafer",
#         r"tsmc",
#         r"chip act",
#         r"chips act",
#         r"chip foundry",
#         r"semiconductor foundry",
#         r"wafer foundry",
#         r"\d+nm.*node|\d+nm.*process|node.*\d+nm",
#         r"nand flash",
#         r"dram shortage",
#         r"export control.*chip|chip.*export control",
#         r"semiconductor.*shortage|shortage.*semiconductor",
#         r"semiconductor.*supply chain",
#     ],
#     "Fed_Interest_Rates": [
#         r"federal reserve",
#         r"\bfed\b.*rate|rate.*\bfed\b",
#         r"fed.*rate hike|us.*rate hike|federal.*rate hike",
#         r"fed.*rate cut|us.*rate cut|federal.*rate cut",
#         r"us interest rate hike|us interest rate cut|fed.*interest rate",
#         r"fomc",
#         r"jerome powell",
#         r"\bpowell\b.*fed",
#         r"fed.*monetary policy|monetary policy.*federal reserve",
#         r"fed.*quantitative easing|federal reserve.*quantitative easing",
#         r"fed.*quantitative tightening|federal reserve.*quantitative tightening",
#         r"fed.*balance sheet",
#         r"fed.*taper|taper.*fed",
#         r"fed funds rate",
#         r"fed.*yield curve|yield curve.*fed|us.*yield curve",
#         r"inverted yield",
#         r"fed.*rate.*pause|rate.*pause.*fed",
#         r"pivot.*fed|fed.*pivot",
#     ],
# }

# # ── Minimum number of DISTINCT patterns that must match to tag an article ─────
# # Counting distinct pattern hits (not raw occurrences) requires the article to
# # exhibit multiple different keyword signals. This only holds because mirrored
# # A.*B / B.*A pairs are collapsed above — reintroducing one as two entries
# # quietly reduces the bar to a single co-occurrence.
# THRESHOLDS = {
#     "AI": 2,
#     "Covid19": 2,
#     "USA_Tariff": 2,
#     "Russia_Ukraine": 2,
#     "Oil_Crisis": 2,
#     "Semiconductor": 2,
#     "Fed_Interest_Rates": 2,
# }

# compiled_individual = {
#     theme: [re.compile(p, re.IGNORECASE) for p in patterns]
#     for theme, patterns in THEMES.items()
# }


# def tag_article(row):
#     text = " ".join(
#         filter(
#             None,
#             [
#                 str(row.get("title", "") or ""),
#                 str(row.get("description", "") or ""),
#                 str(row.get("maintext", "") or ""),
#             ],
#         )
#     )
#     result = {}
#     for theme, patterns in compiled_individual.items():
#         n_matched = sum(1 for p in patterns if p.search(text))
#         result[theme] = n_matched >= THRESHOLDS[theme]
#     return result


# print("Tagging articles...")
# tags = combined.apply(tag_article, axis=1, result_type="expand")
# combined = pd.concat([combined, tags], axis=1)
# print("Done.")

# print("\nTheme coverage (articles where ≥ threshold distinct patterns matched):")
# for theme in THEMES:
#     n = combined[theme].sum()
#     t = THRESHOLDS[theme]
#     print(
#         f"  {theme:<20} {n:>8,} articles  ({n/len(combined)*100:.1f}%)  [threshold={t}]"
#     )

# multi_tagged = (combined[list(THEMES.keys())].sum(axis=1) > 1).sum()
# print(f"\nArticles with 2+ themes: {multi_tagged:,}")
# print(
#     f"Articles with no theme : {(combined[list(THEMES.keys())].sum(axis=1) == 0).sum():,}"
# )

# combined["themes"] = combined[list(THEMES.keys())].apply(
#     lambda row: ",".join(t for t in THEMES if row[t]), axis=1
# )

In [ ]:
# Drop all price columns
price_cols = [
    c
    for c in combined.columns
    if c.endswith(("_day_price_" + c.split("_")[-1]))
    or any(
        c.startswith(p)
        for p in ("prev_day_price_", "curr_day_price_", "next_day_price_")
    )
]

print(f"Dropping {len(price_cols)} price columns:")
print(price_cols)

combined.drop(columns=price_cols, inplace=True)
print(f"\nRemaining columns: {combined.shape[1]}")

In [ ]:
# add unique identifier for articles
combined.insert(
    0,
    "article_id",
    combined["url"].apply(lambda u: hashlib.md5(u.encode()).hexdigest()[:12]),
)

In [ ]:
# body_md5 = (
#     combined["maintext"]
#     .fillna("")
#     .map(lambda t: hashlib.md5(str(t).encode("utf-8", "ignore")).hexdigest())
# )

# n_before = len(combined)
# if {"knowledge_graph", "article_id"}.issubset(combined.columns):
#     # post-extraction frame: protect gold parents and rows that carry a KG
#     parents = {
#         r["parent_article_id"]
#         for r in json.loads(Path("synthetic_results.json").read_text())
#     }
#     order = pd.DataFrame(
#         {
#             "is_parent": (~combined["article_id"].isin(parents)).astype(int),
#             "no_kg": combined["knowledge_graph"].isna().astype(int),
#         }
#     ).sort_values(["is_parent", "no_kg"], kind="stable")
#     keep_idx = body_md5.reindex(order.index).reset_index().groupby(0)["index"].first()
#     combined = combined.loc[sorted(keep_idx)].reset_index(drop=True)
# else:
#     combined = combined[~body_md5.duplicated(keep="first")].reset_index(drop=True)

# print(
#     f"deduplicated on body: {n_before:,} -> {len(combined):,} "
#     f"({n_before - len(combined):,} removed)"
# )

In [ ]:
out_path = data_dir / "combined_2017_2023_cleaned.csv"
combined.to_csv(out_path, index=False)
print(f"Saved to {out_path}")